# AnyText Document Feasibility (Colab GPU) - launcher

Chạy lần lượt 4 bước. Không cần dán thêm cell vá; mã nguồn được tải từ commit đã khóa và kiểm tra SHA-256.

1. **Chuẩn bị và kiểm tra môi trường** (Python 3.10 cô lập, `pip check`, smoke test GPU fp16).
2. **Nhận archive đầu vào và preflight** (tải lên `document_feasibility_results.zip`).
3. **Chạy hai attempts** (A: CORD row 22, B: CORD row 61; hiển thị tiến độ, log đầy đủ lưu riêng).
4. **Xem kết quả và tải archive** của đúng lần chạy hiện tại.

Nguồn: `nomozer/forensics-web-lab` @ `80ecfd72abc8cbe21586101bdee2483b60663a50`. Cần runtime GPU (T4 trở lên), không cần token/đăng nhập.
Human Content QC luôn ở trạng thái `PENDING_HUMAN_REVIEW`.

In [ ]:
# Bước 1: Chuẩn bị và kiểm tra môi trường
import hashlib, pathlib, subprocess, sys
COMMIT = '80ecfd72abc8cbe21586101bdee2483b60663a50'
PINNED = {
    "scripts/research/run_anytext_colab.py": "3f1b46a222dc80db3e0750b1e6f8040565e01d43ba38ce4677bc5893b727a593",
    "scripts/research/anytext_colab_setup.py": "2eedd984e5b0c9999bcad874c9625e443abd7edef4f04b6ec7b58236a37c9623",
    "scripts/research/anytext_requirements.txt": "0464a556e8c99ca5af240985e7d85c9bfa91e4ee3821ba228d5c17ac48d9e02c",
    "ml/evaluation/anytext_document_adapter.py": "d30342c40997f0a15c9a1a489232ad35399ba354761bf71cf5354ec46334a0c5",
    "ml/evaluation/document_feasibility_geometry.py": "566d10670570a0f89f2dbde3dd56092866b1b632fe5f7a109c879399ac0c4c73",
    "research/evidence/document_pilot/anytext_feasibility_plan.json": "88f7e32873ff4c4e96dc60aae33a50aee677a550291e3316a027e1584e69b3ad"
}
REPO = pathlib.Path('/content/forensics-web-lab')
SETUP = REPO / 'scripts/research/anytext_colab_setup.py'

def sh(cmd, cwd=None):
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    rc = p.wait()
    assert rc == 0, f'{cmd[:3]} failed with exit code {rc}'

REPO.mkdir(parents=True, exist_ok=True)
if not (REPO / '.git').is_dir():
    sh(['git', 'init', '-q'], cwd=REPO)
    sh(['git', 'remote', 'add', 'origin', 'https://github.com/nomozer/forensics-web-lab.git'], cwd=REPO)
sh(['git', 'fetch', '-q', '--depth', '1', '--filter=blob:none', 'origin', COMMIT], cwd=REPO)
sh(['git', 'sparse-checkout', 'set', '--cone', 'scripts/research', 'ml/evaluation', 'research/evidence/document_pilot'], cwd=REPO)
sh(['git', 'checkout', '-q', '-f', COMMIT], cwd=REPO)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == COMMIT, f'commit mismatch: {head}'
for rel, sha in PINNED.items():
    assert hashlib.sha256((REPO / rel).read_bytes()).hexdigest() == sha, f'SHA-256 mismatch: {rel}'
print(f'[PASS] {len(PINNED)} source files verified at {COMMIT[:12]}')
sh([sys.executable, '-u', str(SETUP), 'env', '--repo-root', str(REPO)])

In [ ]:
# Bước 2: Nhận archive đầu vào và preflight
import hashlib, pathlib
from google.colab import files
ARCHIVE = pathlib.Path('/content/document_feasibility_results.zip')
ARCHIVE_SHA256 = 'deadd9205207eaaa296a8d9eeb493ee48eeea76ee7afd210684187e4026ed684'
if not ARCHIVE.is_file():
    print('Tải lên document_feasibility_results.zip')
    for name, data in files.upload().items():
        if name.endswith('.zip'):
            ARCHIVE.write_bytes(data)
assert ARCHIVE.is_file() and hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == ARCHIVE_SHA256, 'archive SHA-256 mismatch'
sh([sys.executable, '-u', str(SETUP), 'run', '--archive', str(ARCHIVE), '--preflight-only'])

In [ ]:
# Bước 3: Chạy hai attempts (A và B)
import time
RUN_STARTED = time.time()
sh([sys.executable, '-u', str(SETUP), 'run', '--archive', str(ARCHIVE), '--label', 'repair_fp16_xformers'])

In [ ]:
# Bước 4: Xem kết quả và tải archive của lần chạy hiện tại
import datetime, hashlib, json, pathlib
from IPython.display import Image, display
from google.colab import files
out = pathlib.Path('/content/outputs')
ptr = json.loads((out / 'latest_run.json').read_text(encoding='utf-8'))
assert datetime.datetime.fromisoformat(ptr['timestamp']).timestamp() >= RUN_STARTED, 'latest_run.json belongs to an older run'
sheet = out / ptr['run_dir'] / 'anytext_contact_sheet.png'
if sheet.is_file():
    display(Image(str(sheet)))
zip_path = out / ptr['archive']
assert hashlib.sha256(zip_path.read_bytes()).hexdigest() == ptr['archive_sha256'], 'archive hash mismatch'
print(f"Run {ptr['run_id']}: {ptr['status']} -> {zip_path.name} ({zip_path.stat().st_size / 2**20:.1f} MiB)")
files.download(str(zip_path))